In [1]:
from IPython.display import clear_output
!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()
import os
os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [2]:
from pathlib import Path
import re
import joblib
from IPython.display import clear_output
from datasets import load_dataset
import einops
from typing import cast

from collections import defaultdict
import torch
from utils.data import (
    extract_user_instruction,
)

from mech_interp_toolkit.utils import get_layer_components, load_model_tokenizer_config
from mech_interp_toolkit.activation_utils import get_activations, get_embeddings_dict

In [3]:
def extract_layer_number(filepath: str) -> int:
    filename = Path(filepath).stem

    match = re.search(r'layer_(\d+)', filename)

    if match:
        return int(match.group(1))
    else:
        raise ValueError(f"Could not extract layer number from filename: {filepath}")

In [4]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_train"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "suffix.pt"

batch_size = 32
probe_layer = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
probe_base_dir = Path(".")

for probe_file in probe_base_dir.glob("*.joblib"):
    layer = extract_layer_number(str(probe_file))
    if layer != probe_layer:
        continue
    probe = joblib.load(probe_file)

probe_dir = torch.from_numpy(probe["model"].coef_).to("cuda:0").squeeze().to(torch.bfloat16)
probe_dir = probe_dir/probe_dir.norm()

In [8]:
# Load 200 examples from the circuit_breakers_train split
dataset = load_dataset(
    dataset_name,
    split=split,
)

dataset = cast(dict, dataset)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]

clear_output()

# Load model, tokenizer and config
model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",  # Scaled Dot Product Attention for efficiency
)

clear_output()


# load suffix
# from rashad's eval code
def load_suffix(suffix_path: str, device: torch.device):
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb


suffix_embed = load_suffix(suffix_path=suffix_path, device=device)
len_suffix = suffix_embed.shape[1]

In [9]:
components = get_layer_components(model, stop_at=probe_layer, include_block_outputs=False )

In [ ]:
# Iterate over prompts_str in batches
num_batches = (len(prompts_str) + batch_size - 1) // batch_size

base_dla_collate = []
new_dla_collate = []

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(prompts_str))

    batch_prompts = prompts_str[start_idx:end_idx]

    current_batch_size = len(batch_prompts)

    print(
        f"Processing batch {batch_idx + 1}/{num_batches} (samples {start_idx} to {end_idx - 1})"
    )

    # Use torch.no_grad() to prevent gradient accumulation
    with torch.no_grad():
        batch_dict = ch_tokenizer(prompts=batch_prompts)
        # removes "input_ids" and adds "inputs_embeds"
        batch_embeds_dict = get_embeddings_dict(model, batch_dict)
        batch_embeds = batch_embeds_dict["inputs_embeds"]
        batch_attn_mask = batch_embeds_dict["attention_mask"]

        # broadcast suffix
        batch_suffix = einops.repeat(
            suffix_embed,
            "dummy pos d_model -> (curr_batch dummy) pos d_model",
            curr_batch=current_batch_size,
        )

        new_embeds = torch.cat(
            [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
        )
        attn_extension = torch.ones((current_batch_size, len_suffix), device=device)
        new_attn = torch.cat([batch_attn_mask, attn_extension], dim=1)

        new_embeds_dict = {"inputs_embeds": new_embeds, "attention_mask": new_attn}

        base_acts = get_activations(
            model,
            inputs=batch_embeds_dict,
            layer_components=components,
            retain_grads=False,
            positions=-1,
        ).apply(torch.squeeze)

        new_acts = get_activations(
            model,
            inputs=new_embeds_dict,
            layer_components=components,
            retain_grads=False,
            positions=-1,
        ).apply(torch.squeeze)

        # Compute DLA and immediately move to CPU
        base_dla = (base_acts @ probe_dir).cpu()
        new_dla = (new_acts @ probe_dir).cpu()

        base_dla_collate.append(base_dla)
        new_dla_collate.append(new_dla)
    
    # Clear GPU cache after each batch
    del batch_dict, batch_embeds_dict, batch_embeds, batch_attn_mask
    del batch_suffix, new_embeds, attn_extension, new_attn, new_embeds_dict
    del base_acts, new_acts, base_dla, new_dla
    torch.cuda.empty_cache()
    
    # Optional: print memory usage
    if batch_idx % 5 == 0:
        print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
from tqdm import tqdm
save_path = Path(f"outputs/dla_on_probes/{split}_last_pos.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)

for i, z in enumerate(tqdm(zip(base_dla_collate, new_dla_collate))):
    torch.save(z, save_path.parent / (f"{i}_" + str(save_path.name)))